# 03.03 · Long conversations: trim, delete, summarize

**Module 03 · Memory & Context Engineering** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 03.02 (checkpointers). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Measure** how many tokens a thread sends to the model on every turn, and why that number matters.
2. **Trim** with `trim_messages`, either **transiently** (only what the model sees) or **permanently** (the saved state).
3. **Delete** messages with `RemoveMessage` without breaking tool-call pairs.
4. **Summarize** old history with `SummarizationMiddleware` (`trigger`, `keep`, `token_counter`).
5. **Compare** the three strategies by tokens sent and by what the agent still remembers.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Short-term memory → Common patterns](https://docs.langchain.com/oss/python/langchain/short-term-memory#common-patterns) | **Main page.** Trim, delete and summarize in agents |
| [Built-in middleware → Summarization](https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization) | `SummarizationMiddleware` options: `trigger`, `keep`, `token_counter`, `summary_prompt` |
| [Messages](https://docs.langchain.com/oss/python/langchain/messages) | Message types and ids (`RemoveMessage` needs ids) |
| [Groq rate limits](https://console.groq.com/docs/rate-limits) | Tokens-per-minute limits: why long histories hurt on the free tier |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/03-memory-and-context/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence one known, harmless warning (same as Lesson 02.04).
# langchain-core 1.6.9 and 1.6.10 serialize the injected ToolRuntime/Runtime while parsing input and Pydantic
# warns "PydanticSerializationUnexpectedValue ... field_name='context'". Context still arrives intact.
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · The problem, in numbers

With a checkpointer, **every** turn sends the **whole** thread to the model. Turn 20 pays for turns 1–19 again. Three costs grow:

- **Tokens and money**, and on Groq's free tier, your **tokens-per-minute** limit (a long thread hits `429` sooner).
- **Latency**: more input, slower first token.
- **Quality**: models get distracted by stale content long before the context window is full (the short-term memory page says it plainly).

`mastery.memory.count_tokens` is `count_tokens_approximately` (≈ 4 characters per token + a few per message), the same counter `SummarizationMiddleware` uses by default. It's an estimate; `usage_metadata` from Groq is the real number.

In [ ]:
# 🧪 Offline: a 12-turn tutoring chat. The scripted model records what it was sent on each call.
from langchain.agents import create_agent
from mastery.memory import count_tokens, make_checkpointer, thread_config
from mastery.testing import ScriptedChatModel

QUESTIONS = [
    "Hi, I'm Ana. I'm 14 and I'm preparing for a math exam on Friday.",
    "Can you explain what a fraction is?",
    "Why is 2/4 the same as 1/2?",
    "How do I add 1/3 and 1/4?",
    "What is a common denominator?",
    "Can you give me a harder example?",
    "How do I subtract fractions?",
    "What about mixed numbers like 2 1/2?",
    "How do I multiply fractions?",
    "And dividing?",
    "Give me one practice problem.",
    "What's my name and when is my exam?",
]
TUTOR_REPLY = ("Good question! Think of a pizza cut into equal slices: the bottom number says how many slices, "
               "the top number says how many you take. Let's work through it step by step together. ")


def tutor_model() -> ScriptedChatModel:
    return ScriptedChatModel(responses=[TUTOR_REPLY * 2], repeat_last=True)


def run_chat(agent, model, thread: str) -> list[int]:
    """Ask every question in one thread; return the tokens the model received on each turn."""
    start = len(model.calls)
    for q in QUESTIONS:
        agent.invoke({"messages": [{"role": "user", "content": q}]}, thread_config(thread))
    return [count_tokens(call) for call in model.calls[start:]]


plain_model = tutor_model()
plain = create_agent(plain_model, tools=[], checkpointer=make_checkpointer("memory"))
plain_tokens = run_chat(plain, plain_model, "plain")
print("tokens sent per turn:", plain_tokens)
print("total sent          :", sum(plain_tokens))

**You should now see** the per-turn count climbing by ~107 tokens every turn (from 20 to about 1,200) and a total around 7,300. That's for a short chat with short answers. Real tutor replies are longer, and tool results longer still.

---
## Part 2 · Trimming with `trim_messages`

`trim_messages(messages, max_tokens=..., token_counter=..., strategy="last", start_on="human", include_system=True)` keeps the most recent messages that fit in `max_tokens`:

| Argument | Why |
|---|---|
| `strategy="last"` | Keep the newest messages (the default) |
| `token_counter=count_tokens_approximately` | Or a model (`llm`) for its own count, or the string `"approximate"` |
| `start_on="human"` | Most providers want the history to start with a user message |
| `end_on=("human", "tool")` | Optional: the history should end where the model must answer |
| `include_system=True` | Never trim away the system prompt |

It returns a **new list**; your state is unchanged. That makes it the tool for **transient** trimming.

In [ ]:
# 🧪 Offline: trim the 'plain' thread to ~300 tokens.
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately

history = plain.get_state(thread_config("plain")).values["messages"]
kept = trim_messages(history, max_tokens=300, token_counter=count_tokens_approximately,
                     strategy="last", start_on="human", include_system=True)
print(f"history: {len(history)} messages, {count_tokens(history)} tokens")
print(f"trimmed: {len(kept)} messages, {count_tokens(kept)} tokens; first kept: {kept[0].content[:45]!r}")

**You should now see** `history: 24 messages, …` and `trimmed: 4 messages, …` (the last two turns) starting with `'Give me one practice problem.'`: everything older, including Ana's name and exam day, is gone from what the model would see.

### Transient vs permanent trimming

| | **Transient** (`@wrap_model_call`) | **Permanent** (`@before_model`) |
|---|---|---|
| What changes | Only the model **request** (`request.override(messages=...)`) | The **state**: `RemoveMessage(id=REMOVE_ALL_MESSAGES)` + the kept messages |
| Saved thread | Keeps the full history | Loses the old messages for good |
| Use when | You may need the history later (audit, UI, summaries) | Storage and privacy matter more |

The docs' trimming example uses `@before_model` (permanent). Lesson 03.05 calls the transient kind "model context" engineering.

In [ ]:
# 🧪 Offline: both styles as middleware, measured on the same 12 questions.
from typing import Any

from langchain.agents import AgentState
from langchain.agents.middleware import ModelRequest, before_model, wrap_model_call
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime

BUDGET = 300


def _trim(messages):
    return trim_messages(messages, max_tokens=BUDGET, token_counter=count_tokens_approximately,
                         strategy="last", start_on="human", include_system=True)


@wrap_model_call
def trim_for_model(request: ModelRequest, handler):
    """Transient: the model sees a trimmed copy; the thread keeps everything."""
    return handler(request.override(messages=_trim(request.messages)))


@before_model
def trim_state(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Permanent: replace the saved messages with the trimmed ones."""
    kept = _trim(state["messages"])
    if len(kept) == len(state["messages"]):
        return None
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *kept]}


results = {"no trimming": (plain_tokens, plain)}
for name, mw in [("transient trim", trim_for_model), ("permanent trim", trim_state)]:
    model = tutor_model()
    agent = create_agent(model, tools=[], middleware=[mw], checkpointer=make_checkpointer("memory"))
    results[name] = (run_chat(agent, model, name), agent)

print(f"{'strategy':<15} {'total sent':>10} {'max/turn':>9} {'saved msgs':>11}")
for name, (tokens, agent) in results.items():
    saved = len(agent.get_state(thread_config(name if name != "no trimming" else "plain")).values["messages"])
    print(f"{name:<15} {sum(tokens):>10,} {max(tokens):>9,} {saved:>11}")

**You should now see** that both trims cut the total to about a third (≈2,400 vs ≈7,300) and keep every turn under 300 tokens. The **saved** message count is the difference: 24 for no trimming and transient trimming, 6 for permanent trimming.

### What just happened?

- `wrap_model_call` wraps the model node: it receives the `ModelRequest`, changes a copy with `override`, and calls `handler`. State is untouched.
- `before_model` runs as its own step before the model and **returns a state update**. `RemoveMessage(id=REMOVE_ALL_MESSAGES)` tells the `add_messages` reducer to clear the list, then the kept messages are added back.
- Neither strategy can answer question 12 (*"What's my name and when is my exam?"*): the facts were in message 1. Trimming saves tokens by **forgetting**. Summaries (Part 4) and long-term memory (03.04) are how you keep facts.

---
## Part 3 · Deleting messages safely

`RemoveMessage(id=m.id)` deletes one message (it needs a state key with the `add_messages` reducer; `AgentState.messages` has it). The danger is an **invalid history**:

- An `AIMessage` with `tool_calls` must be followed by a `ToolMessage` for **each** call. Delete one without the other and the next model call fails (Groq: `400 ... tool_call_id ...`).
- Many providers want the conversation (after the system prompt) to start with a user message.

The safe rule: cut only at a **human message** boundary, so tool-call pairs stay together.

In [ ]:
# 🧪 Offline: a safe "delete everything before the last N user turns" helper.
from langchain.messages import AIMessage, HumanMessage, ToolMessage


def removals_keeping_last_turns(messages, turns: int) -> list[RemoveMessage]:
    """RemoveMessage for everything before the last `turns` human messages (never splits a tool pair)."""
    human_positions = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]
    if len(human_positions) <= turns:
        return []
    cut = human_positions[-turns]
    return [RemoveMessage(id=m.id) for m in messages[:cut] if m.type != "system"]


demo = [
    HumanMessage("What is 3/4 of 20?", id="h1"),
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "3/4*20"}, "id": "c1"}], id="a1"),
    ToolMessage("15", tool_call_id="c1", id="t1"),
    AIMessage("3/4 of 20 is 15.", id="a2"),
    HumanMessage("And 2/5 of 20?", id="h2"),
    AIMessage("8.", id="a3"),
]
print("remove:", [r.id for r in removals_keeping_last_turns(demo, turns=1)])

naive = demo[2:]  # "drop the first two messages", like the docs' minimal example
orphan = [m.id for m in naive if isinstance(m, ToolMessage)]
print("naive cut starts with:", type(naive[0]).__name__, "| orphan tool results:", orphan)

**You should now see** `remove: ['h1', 'a1', 't1', 'a2']` (the whole first turn, tool pair included), and that a naive "drop the first two" leaves `ToolMessage` `t1` without its `AIMessage`, an invalid history.

---
## Part 4 · Summarizing with `SummarizationMiddleware`

Summaries keep the **gist** of old turns instead of dropping them. The built-in middleware runs before each model call:

1. Count tokens of the state's messages (`token_counter`, default `count_tokens_approximately`).
2. If a **`trigger`** is reached, e.g. `("tokens", 600)` or `("messages", 10)` (a list means *any*; a dict means *all*), call the summary **`model`** on the older messages.
3. Replace them, **in the state**, with one summary message and keep the recent ones (**`keep`**, default `("messages", 20)`). It never splits a tool-call pair.

`("fraction", 0.8)` (80% of the context window) works only when the **summary model** has a profile with `max_input_tokens`. Groq models from `get_model` do (`openai/gpt-oss-20b`: 131,072, checked with langchain-groq 1.1.3); our scripted model doesn't, and the constructor raises `ValueError`. Offline we use `tokens`. For a free-tier budget, `tokens` is clearer anyway: 80% of 131k is far above Groq's tokens-per-minute limit.

Offline, a second scripted model plays the summarizer so we can watch the mechanics.

In [ ]:
# 🧪 Offline: summarize when the thread passes 600 tokens; keep the last 4 messages.
from langchain.agents.middleware import SummarizationMiddleware

summary_model = ScriptedChatModel(
    responses=["Learner: Ana, 14. Exam on Friday. Covered: what fractions are, equivalence, adding with "
               "common denominators, subtracting, mixed numbers, multiplying. Explain with pizza analogies."],
    repeat_last=True)
sum_model = tutor_model()
summarizing = create_agent(
    sum_model, tools=[], checkpointer=make_checkpointer("memory"),
    middleware=[SummarizationMiddleware(model=summary_model, trigger=("tokens", 600), keep=("messages", 4))])
sum_tokens = run_chat(summarizing, sum_model, "summarized")

final = summarizing.get_state(thread_config("summarized")).values["messages"]
print("tokens sent per turn:", sum_tokens)
print("summaries made      :", len(summary_model.calls))
print("saved messages      :", len(final), "| first one:", type(final[0]).__name__, final[0].additional_kwargs)
print(final[0].content[:160])

**You should now see** per-turn tokens that grow, drop when a summary is made, and grow again (a saw-tooth that stays under 600), `summaries made : 2`, and a saved state that starts with a `HumanMessage` marked `{'lc_source': 'summarization'}` whose text begins *"Here is a summary of the conversation to date:"*.

### What just happened?

- The middleware ran in `before_model`. Each time the thread passed 600 tokens, it sent the older messages (wrapped in its `summary_prompt`) to `summary_model` and returned a state update: remove all, add the summary, add back the last 4 messages.
- The summary is a **`HumanMessage`**, so the history still starts with a user turn. `additional_kwargs["lc_source"] == "summarization"` lets you find it (and hide it in a UI).
- Summaries cost extra model calls. Use a **cheap, fast model** for them (`get_model("fast")`), and a trigger that fires rarely.

In [ ]:
# 🧪 Offline: the final comparison, including how much each strategy still "knows" about message 1.
def remembers_exam(agent, thread: str) -> bool:
    text = " ".join(str(m.content) for m in agent.get_state(thread_config(thread)).values["messages"])
    return "Friday" in text


rows = [("no trimming", plain_tokens, plain, "plain"),
        ("transient trim", *results["transient trim"], "transient trim"),
        ("permanent trim", *results["permanent trim"], "permanent trim"),
        ("summarization", sum_tokens, summarizing, "summarized")]
print(f"{'strategy':<15} {'total sent':>10} {'max/turn':>9} {'last turn':>10}  exam day still in state?")
for name, tokens, agent, thread in rows:
    print(f"{name:<15} {sum(tokens):>10,} {max(tokens):>9,} {tokens[-1]:>10,}  {remembers_exam(agent, thread)}")

**You should now see** that summarization sends far fewer tokens than no trimming, a bit more than trimming, and is the only budget-limited strategy where "Friday" survives in the state. (Transient trimming keeps "Friday" in the *state*, but the model never *sees* it.)

In [ ]:
# 💸 Live: a real summary of the 'plain' thread with the fast model (1 call, ~1.5k input tokens).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_summarizer = SummarizationMiddleware(model=get_model("fast", temperature=0, reasoning_effort="low"),
                                              trigger=("tokens", 600), keep=("messages", 4))
    live_tutor_model = tutor_model()
    live_agent = create_agent(live_tutor_model, tools=[], middleware=[live_summarizer],
                              checkpointer=make_checkpointer("memory"))
    # Seed the thread with the long history, then one more turn triggers a real summary.
    live_agent.update_state(thread_config("live"), {"messages": history})
    live_agent.invoke({"messages": [{"role": "user", "content": "What's my name and when is my exam?"}]},
                      thread_config("live"))
    summary = live_agent.get_state(thread_config("live")).values["messages"][0]
    print(summary.content[:1200])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** a structured summary (the default prompt asks for *SESSION INTENT*, *SUMMARY*, *ARTIFACTS*, *NEXT STEPS*) that mentions Ana, the Friday exam and the fraction topics. Only the summary is live here; the tutor is still scripted, to keep the cost to one call.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `400 ... tool_call_id ... not found` / *tool messages must follow tool calls* | A delete or trim split an `AIMessage(tool_calls)` from its `ToolMessage` | Cut at human-message boundaries (Part 3); `trim_messages(..., start_on="human")`; the summarization middleware does this for you |
| `ValueError: Model profile information is required to use fractional token limits` | `("fraction", x)` with a summary model that has no `max_input_tokens` profile | Use `("tokens", n)` / `("messages", n)`, or build the model with `profile={"max_input_tokens": ...}` |
| Summarization never runs | No `trigger` (it's `None` by default) | Always set `trigger=` |
| Summaries on every turn, slow and costly | `trigger` lower than what `keep` leaves behind | Make `keep` clearly smaller than `trigger` |
| Trimmed agent forgets the user's name | Trimming forgets by design | Summarize, or store facts in long-term memory (03.04) |
| `RemoveMessage` does nothing | Message has no `id`, or the key has no `add_messages` reducer | Use messages from the state (they have ids); use `AgentState` |
| Token counts differ from Groq's bill | `count_tokens_approximately` is an estimate | Compare with `usage_metadata` (Lesson 01.07); keep a margin |

## ⚠️ Production notes

> ⚠️ **Production note: budget in tokens, not messages.**
> One tool result can be 10,000 tokens. Trigger on `("tokens", n)` with `n` well below your model's input limit **and** your Groq tokens-per-minute limit, and cap tool output size at the tool (Lesson 02.04).

> ⚠️ **Production note: summaries can drop or invent facts.**
> A summary is a model output. Keep identity, permissions and important facts **out of** the summary path: put them in runtime context (03.05) or the store (03.04). Log summaries so you can audit what the agent "remembers".

> ⚠️ **Production note: transient trimming keeps an audit trail.**
> If you must show users their full chat or keep it for compliance, trim what the model **sees** (`wrap_model_call`) and keep the saved thread intact, then apply a retention policy to the thread itself.

## 🏋️ Exercises

1. **(Easy)** Change `BUDGET` to 150 and 600. How do total tokens and the number of saved messages change?
2. **(Medium)** Make `SummarizationMiddleware` trigger on `[("tokens", 800), ("messages", 10)]`. Which condition fires first in our chat?
3. **(Medium)** Write a `@before_model` middleware that uses `removals_keeping_last_turns(..., turns=3)`. Test it on a scripted chat **with tool calls** and check that the next model call doesn't receive an orphan `ToolMessage`.
4. **(Stretch)** Write your own `summary_prompt` (it must contain `{messages}`) that outputs only *Learner facts* and *Topics covered*. Compare its length with the default prompt's output in the 💸 cell.

<details>
<summary><b>Show solutions</b></summary>

**2.** With our short messages, `("messages", 10)` usually fires first (10 messages are ~5 turns, < 800 tokens). A list means "any condition".

**3.**
```python
@before_model
def keep_three_turns(state, runtime):
    removals = removals_keeping_last_turns(state["messages"], turns=3)
    return {"messages": removals} if removals else None
```
Then assert: for every `ToolMessage` in `model.calls[-1]`, an earlier `AIMessage` in the same list has a matching `tool_calls[*]["id"]`.
</details>

## ✅ Checkpoint

- [ ] I can measure tokens per turn and explain the three costs of long threads.
- [ ] I can trim transiently (`wrap_model_call`) and permanently (`before_model`), and choose between them.
- [ ] I can delete messages without breaking tool-call pairs.
- [ ] I can configure `SummarizationMiddleware` with `trigger`, `keep` and a cheap model.
- [ ] I can compare strategies by tokens **and** by what's remembered.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

The built-in middleware page also lists **context editing** (clearing old tool outputs). Read it and note: which messages does it change, which trigger does it use, and when would you use it instead of summarization? Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Every turn resends the thread: tokens, latency and quality all get worse as it grows.
- `trim_messages` keeps the newest messages that fit; use it transiently (request) or permanently (state).
- `RemoveMessage` deletes by id; cut at human-message boundaries to keep histories valid.
- `SummarizationMiddleware` replaces old turns with a summary message in the state, triggered by tokens or messages.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 03.04 · Long-term memory with stores** → [`03-04-long-term-memory.ipynb`](03-04-long-term-memory.ipynb).

[Module 03 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)